# RAG-Augmented Distillation — Full GPU Run

This notebook runs the complete RAD pipeline end-to-end on a T4 GPU (Colab) or 2×T4 (Kaggle).

**Estimated time on Colab T4:**
- Phase 1 — Build ChromaDB: ~15 min (CPU, embedder only)
- Phase 2 — Generate soft labels: ~30 min (GPU, teacher forward passes)
- Phase 3 — Train student: ~45 min (GPU, 10k examples × 3 epochs)
- Phase 4 — Evaluate: ~10 min (GPU)

**All outputs are persisted to Google Drive so they survive Colab disconnections.**

## Cell 1 — GPU info & install

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Go to Runtime > Change runtime type > T4 GPU.")

# Clone repo if not already present
import os
if not os.path.exists('/content/model-distillation-'):
    !git clone https://github.com/aabhimittal/model-distillation- /content/model-distillation-

os.chdir('/content/model-distillation-')
!pip install -e '.[dev]' -q

## Cell 2 — Mount Google Drive & set paths

Outputs go to Drive so they survive Colab disconnections.
Skip the mount cell if running on Kaggle (use `/kaggle/working/` instead).

In [ ]:
# --- Colab: mount Drive ---
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/rad-distillation'
    print(f'Using Google Drive: {BASE_DIR}')
except ImportError:
    # Kaggle or local
    BASE_DIR = '/kaggle/working/rad-distillation'
    print(f'Not in Colab. Using: {BASE_DIR}')

import os
CHROMA_DIR     = f'{BASE_DIR}/chroma_db'
SOFT_LABELS_DIR = f'{BASE_DIR}/soft_labels'
OUTPUT_DIR     = f'{BASE_DIR}/outputs'

for d in [CHROMA_DIR, SOFT_LABELS_DIR, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'ChromaDB    : {CHROMA_DIR}')
print(f'Soft labels : {SOFT_LABELS_DIR}')
print(f'Outputs     : {OUTPUT_DIR}')

## Cell 3 — Phase 1: Build ChromaDB vector store

Embeds ~50k SQuAD Wikipedia passages into ChromaDB using MiniLM-L6-v2.
**Idempotent** — safe to re-run; skips if DB already populated.

In [ ]:
!python scripts/build_vector_db.py \
    --config configs/distillation_config.yaml \
    --chroma-dir {CHROMA_DIR}

## Cell 4 — Phase 2: Generate soft labels

Runs the teacher (flan-t5-base) with and without retrieved context over all 10k
training examples. Saves three logit tensors per example to `{SOFT_LABELS_DIR}/`.
**Idempotent** — skips already-generated examples.

For **Kaggle 2×T4**, add `--device-map auto` to distribute the teacher across both GPUs.

In [ ]:
# Standard (Colab T4):
!python scripts/generate_soft_labels.py \
    --config configs/distillation_config.yaml \
    --soft-labels-dir {SOFT_LABELS_DIR} \
    --chroma-dir {CHROMA_DIR}

# Kaggle 2×T4 (uncomment):
# !python scripts/generate_soft_labels.py \
#     --config configs/distillation_config.yaml \
#     --soft-labels-dir {SOFT_LABELS_DIR} \
#     --chroma-dir {CHROMA_DIR} \
#     --device-map auto

In [ ]:
# Quick check — how many .npz files were generated?
import glob
npz_files = glob.glob(f'{SOFT_LABELS_DIR}/*.npz')
print(f'Soft label files: {len(npz_files)}')
if npz_files:
    import numpy as np
    sample = np.load(npz_files[0])
    print(f'Keys: {list(sample.keys())}')
    print(f'rag_logits shape: {sample["rag_logits"].shape}')  # expect (L, V)

## Cell 5 — Phase 3: Train the RAD student

Trains flan-t5-small with the full L_RAD loss for 3 epochs.
Checkpoints are saved to `{OUTPUT_DIR}` every 500 steps.
Loss components (L_RAG, L_KL, L_CRA, L_CE) are logged every 50 steps.

In [ ]:
!python scripts/train_student.py \
    --config configs/distillation_config.yaml \
    --output-dir {OUTPUT_DIR} \
    --soft-labels-dir {SOFT_LABELS_DIR} \
    --chroma-dir {CHROMA_DIR}

In [ ]:
# Plot training loss curves
import json
import matplotlib.pyplot as plt

history_path = f'{OUTPUT_DIR}/loss_history.json'
with open(history_path) as f:
    history = json.load(f)

steps = [h['step'] for h in history]
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(steps, [h['loss'] for h in history], label='Total')
axes[0].set_title('Total Loss')
axes[0].set_xlabel('Step')
axes[0].legend()

for key, label in [('L_RAG', 'L_RAG (novel)'), ('L_KL', 'L_KL (standard KD)'),
                   ('L_CRA', 'L_CRA (contrastive)'), ('L_CE', 'L_CE (hard label)')]:
    axes[1].plot(steps, [h[key] for h in history], label=label)
axes[1].set_title('Loss Components')
axes[1].set_xlabel('Step')
axes[1].legend()

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/loss_curves.png', dpi=150)
plt.show()
print('Loss curves saved.')

## Cell 6 — Phase 4: Evaluate all conditions

In [ ]:
STUDENT_CHECKPOINT = f'{OUTPUT_DIR}/final'

!python scripts/evaluate.py \
    --config configs/distillation_config.yaml \
    --student-rad {STUDENT_CHECKPOINT} \
    --output-dir {OUTPUT_DIR} \
    --chroma-dir {CHROMA_DIR}

## Cell 7 — Results table + bar chart

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np

with open(f'{OUTPUT_DIR}/eval_results.json') as f:
    results = json.load(f)

# Print table
print(f"{'Model':<30} {'EM':>8} {'F1':>8} {'BLEU-4':>8}")
print('-' * 58)
for model_name, metrics in results.items():
    em  = metrics.get('exact_match', 0) * 100
    f1  = metrics.get('f1', 0) * 100
    b4  = metrics.get('bleu4', 0) * 100
    print(f"{model_name:<30} {em:>7.1f}%  {f1:>7.1f}%  {b4:>7.1f}%")

# Bar chart
models = list(results.keys())
em_scores  = [results[m].get('exact_match', 0) * 100 for m in models]
f1_scores  = [results[m].get('f1', 0) * 100 for m in models]

x = np.arange(len(models))
width = 0.35
fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, em_scores, width, label='Exact Match (%)', color='steelblue')
bars2 = ax.bar(x + width/2, f1_scores,  width, label='F1 (%)',          color='coral')

ax.set_xticks(x)
ax.set_xticklabels(models, rotation=15, ha='right')
ax.set_ylabel('Score (%)')
ax.set_title('RAG-Augmented Distillation — Model Comparison (SQuAD v2)')
ax.legend()
ax.bar_label(bars1, fmt='%.1f', padding=3)
ax.bar_label(bars2, fmt='%.1f', padding=3)
ax.set_ylim(0, max(f1_scores) * 1.2)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/results_chart.png', dpi=150)
plt.show()
print(f'Chart saved to {OUTPUT_DIR}/results_chart.png')